# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kuteesatendojeremiah/Tendojerry-Flyrank/blob/main/work/notebooks/w09_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [8]:
%pip install -q duckdb huggingface_hub

import os, getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_clients":       f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content":       f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily":        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    "fact_query_90d":    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Same window as ML-04 through ML-09 (w03-w08) — mid-panel month, never the sealed June 2026 sample.
MONTH_START = "2026-03-01"
MONTH_END_EXCL = "2026-04-01"      # half-open: report_date < MONTH_END_EXCL
PREV30_START = "2026-01-30"        # the 30 days immediately before MONTH_START
PREV30_END_EXCL = MONTH_START

print(f"Connected. Iterating on month={MONTH_START[:7]} | prev30 window: [{PREV30_START}, {PREV30_END_EXCL})")

# --- Rebuild the identical feature vector + label as ML-05/07/08/09, plus content_updated_date for the baseline rule ---
feature_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS imp_prev30,
           SUM(gsc_clicks) AS clk_prev30,
           AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_prev30,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS active_days_prev30
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{PREV30_START}' AND report_date < DATE '{PREV30_END_EXCL}'
    GROUP BY 1, 2
""").df()
feature_frame["ctr_prev30"] = (feature_frame["clk_prev30"] / feature_frame["imp_prev30"]).fillna(0)

content_dates = con.sql(f"SELECT content_hash_id, content_updated_date FROM {TABLES['dim_content']}").df()
feature_frame = feature_frame.merge(content_dates, on="content_hash_id", how="left")

content_schema = con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']}").df()
EXCLUDE_LIKE = ("client", "hash", "id", "profile", "account", "flag", "score")
candidate_cols = [
    c for c in content_schema.loc[content_schema["column_type"] == "VARCHAR", "column_name"]
    if not any(bad in c.lower() for bad in EXCLUDE_LIKE)
]
cat_features = []
for col in candidate_cols:
    n_distinct = con.sql(f"SELECT COUNT(DISTINCT {col}) FROM {TABLES['dim_content']}").fetchone()[0]
    if 1 < n_distinct <= 15:
        cat_features.append(col)

cols_sql = ", ".join(cat_features)
content_meta = con.sql(f"SELECT content_hash_id, {cols_sql} FROM {TABLES['dim_content']}").df()
feature_frame = feature_frame.merge(content_meta, on="content_hash_id", how="left")
for col in cat_features:
    feature_frame[col] = feature_frame[col].fillna("unknown")
feature_frame_dummies = pd.get_dummies(feature_frame, columns=cat_features, prefix=cat_features)

march = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_march
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END_EXCL}'
    GROUP BY 1, 2
""").df()
data = feature_frame_dummies.merge(march, on=["client_hash_id", "content_hash_id"], how="inner")
data = data[data["imp_prev30"] > 0].copy()
data["is_declining"] = (data["imp_march"] < 0.8 * data["imp_prev30"]).astype(int)

feature_cols = [c for c in feature_frame_dummies.columns
                if c not in ("client_hash_id", "content_hash_id", "content_updated_date")]

print(f"\n{len(data):,} content items, {len(feature_cols)} feature columns, base rate {data['is_declining'].mean():.3f}")
print("Same 146,253 rows / 0.285 base rate as ML-05/07/08/09 if the pipeline is consistent.")

# --- Baseline rule, identical to ML-07 (w06_baseline_score.ipynb) ---
update_date = pd.to_datetime(data["content_updated_date"])
month_start_ts = pd.Timestamp(MONTH_START)
valid_update = update_date <= month_start_ts
data["days_since_update"] = np.where(valid_update, (month_start_ts - update_date).dt.days, np.nan)

data["position_bucket"] = pd.cut(
    data["avg_position_prev30"], bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"]
)
ctr_by_position = data.dropna(subset=["position_bucket"]).groupby("position_bucket", observed=True).agg(
    total_clicks=("clk_prev30", "sum"), total_impressions=("imp_prev30", "sum")
)
ctr_by_position["expected_ctr"] = ctr_by_position["total_clicks"] / ctr_by_position["total_impressions"]
data["expected_ctr"] = data["position_bucket"].map(ctr_by_position["expected_ctr"]).astype(float)

VISIBILITY_FLOOR = 100  # below this, a CTR reading is too noisy to trust
visible = data["imp_prev30"] >= VISIBILITY_FLOOR
has_position = data["position_bucket"].notna()
ctr_gap = (data["expected_ctr"] - data["ctr_prev30"]).clip(lower=0)
staleness_multiplier = 1 + data["days_since_update"].fillna(0) / 365  # unknown -> neutral (1x)
data["baseline_score"] = np.where(visible & has_position, ctr_gap * data["imp_prev30"] * staleness_multiplier, 0.0)
data["ctr_gap"] = ctr_gap

# --- Random Forest, identical model/split to ML-08/09, kept ONLY as a secondary corroborating column ---
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

X_full = data[feature_cols].fillna(0)
y_full = data["is_declining"]
groups = data["client_hash_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_full, y_full, groups=groups))
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(
    X_full.iloc[train_idx], y_full.iloc[train_idx]
)
data["rf_probability"] = rf.predict_proba(X_full)[:, 1]

print(f"\nBaseline queue ready: {int((data['baseline_score'] > 0).sum()):,} scoreable items. "
      f"RF probability attached from the same {len(train_idx):,}-row client-grouped training "
      "split as ML-08/09 -- a secondary column only, never used to rank (ML-08 already found "
      "the baseline beats it on Precision@50).")

Paste your Hugging Face READ token (hf_...): ··········
Connected. Iterating on month=2026-03 | prev30 window: [2026-01-30, 2026-03-01)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


146,253 content items, 22 feature columns, base rate 0.285
Same 146,253 rows / 0.285 base rate as ML-05/07/08/09 if the pipeline is consistent.

Baseline queue ready: 54,411 scoreable items. RF probability attached from the same 70,520-row client-grouped training split as ML-08/09 -- a secondary column only, never used to rank (ML-08 already found the baseline beats it on Precision@50).


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Ranker:** ML-07's baseline rule stays the primary ranking function. ML-08 already tested the
alternative honestly — Random Forest under the client-grouped split scored Precision@50 = 0.520
(1.63x), against the baseline's 0.540 (1.69x) — the transparent rule wins on the metric this
repo defends, so it's what actually ranks the queue
(`skills/building-baselines/SKILL.md`: "keep the baseline frozen once the model work starts").
The Random Forest probability rides along as a secondary column, never used to re-rank.

**Reason codes:** ML-07 shipped one flag (`low_ctr_for_stale_position`) for every scored row.
This section expands it into up to three short codes a reviewer can scan without opening the
data:
- `large_ctr_gap` / `small_ctr_gap` — is the CTR-vs-position gap the loud kind or the marginal kind
- `stale_180d_plus` / `recently_updated` / `update_date_unknown` — the staleness signal ML-07
  found genuinely unverifiable for most rows (`226eba1`), so "unknown" is named plainly instead
  of silently defaulted away
- `model_agrees` — appended only when the Random Forest's probability is also >=0.5 on the same
  row: an honest corroboration flag, not a ranking input

**Confidence tiers:** high/medium/low by baseline-score quantile within the scored population
(p80/p50 split) — the same idea `scripts/04_evaluate_and_export.py` uses for the reference
pipeline's queue.

**Action label:** `refresh` when the baseline score is positive, `monitor` otherwise —
unchanged from ML-07, since nothing in this section's evidence supports adding a third action a
human couldn't already infer from the reason codes above.

In [9]:
# --- Confidence tiers: score quantiles within the scored population only ---
scored_mask = data["baseline_score"] > 0
high_threshold = data.loc[scored_mask, "baseline_score"].quantile(0.8)
medium_threshold = data.loc[scored_mask, "baseline_score"].quantile(0.5)

def confidence_label(score):
    if score >= high_threshold:
        return "high"
    if score >= medium_threshold:
        return "medium"
    return "low"

data["confidence"] = np.where(scored_mask, data["baseline_score"].apply(confidence_label), "not_scored")

# --- Reason codes: expand ML-07's single flag into a short, scannable set ---
def reason_codes(row):
    if row["baseline_score"] <= 0:
        return "not_scored"
    codes = []
    gap_pct = row["ctr_gap"] * 100
    codes.append("large_ctr_gap" if gap_pct >= 0.30 else "small_ctr_gap")
    if pd.isna(row["days_since_update"]):
        codes.append("update_date_unknown")
    elif row["days_since_update"] >= 180:
        codes.append("stale_180d_plus")
    else:
        codes.append("recently_updated")
    if row["rf_probability"] >= 0.5:
        codes.append("model_agrees")
    return "|".join(codes)

data["reason_codes"] = data.apply(reason_codes, axis=1)
data["action"] = np.where(scored_mask, "refresh", "monitor")

final_queue = data.sort_values("baseline_score", ascending=False).reset_index(drop=True)
final_queue.insert(0, "rank", final_queue.index + 1)

print("Confidence tier counts:")
print(final_queue["confidence"].value_counts())
print("\nAction counts:")
print(final_queue["action"].value_counts())
print("\nTop 10 rows:")
print(final_queue.head(10)[["rank", "action", "confidence", "reason_codes", "baseline_score", "rf_probability"]])

Confidence tier counts:
confidence
not_scored    91842
low           27204
medium        16324
high          10883
Name: count, dtype: int64

Action counts:
action
monitor    91842
refresh    54411
Name: count, dtype: int64

Top 10 rows:
   rank   action confidence                                    reason_codes  \
0     1  refresh       high               large_ctr_gap|update_date_unknown   
1     2  refresh       high  large_ctr_gap|update_date_unknown|model_agrees   
2     3  refresh       high                  large_ctr_gap|recently_updated   
3     4  refresh       high               large_ctr_gap|update_date_unknown   
4     5  refresh       high               large_ctr_gap|update_date_unknown   
5     6  refresh       high               small_ctr_gap|update_date_unknown   
6     7  refresh       high  large_ctr_gap|update_date_unknown|model_agrees   
7     8  refresh       high               small_ctr_gap|update_date_unknown   
8     9  refresh       high               large_ctr

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who:** an editorial/content team deciding what to review first in a refresh cycle. It is a
triage aid — a sorted list a person still opens and checks — not an auto-publish or
auto-deprioritize system.

**Where it stops being valid:**
- **Staleness is uninformative for most of the queue.** 268,203 of 321,546 items (83%) carry a
  `content_updated_date` after `MONTH_START`, so `update_date_unknown` is the modal reason code,
  not an edge case — a reviewer should not read "unknown" as "old."
- **Validated only at k=50.** Precision@50 is the number this repo defends
  (`skills/building-baselines/SKILL.md`); ranking below roughly rank 50-100 hasn't been checked
  against a held-out label and shouldn't be trusted with the same confidence.
- **One client-grouped test split, one month.** The 0.540/1.69x number comes from 13 held-out
  clients scored against the March-2026 slice. A new client onboarded after this snapshot, or a
  different month, hasn't been validated and needs its own check before this queue is trusted
  for it.
- **Observational, not causal.** Nothing here claims that refreshing a page *causes* recovery —
  see ML-09 Section 4's claim rewrite for the language this repo commits to.

In [10]:
# Numbers that back Section 2's limits, computed from this run rather than asserted
pct_unknown_update = data["days_since_update"].isna().mean() * 100
n_train_clients = groups.iloc[train_idx].nunique()
n_test_clients = groups.iloc[test_idx].nunique()

print(f"Update-date-unknown share: {pct_unknown_update:.1f}% of {len(data):,} scored items")
print(f"Held-out validation covers {n_test_clients} clients (trained on {n_train_clients}) -- "
      f"a client outside these {n_train_clients + n_test_clients} has not been validated against")
print(f"Validation month: {MONTH_START[:7]} only -- no other month has been checked")
print("Precision@50 is the only k this repo defends; ranks below roughly 50-100 aren't "
      "individually validated against a held-out label.")

Update-date-unknown share: 79.7% of 146,253 scored items
Held-out validation covers 13 clients (trained on 29) -- a client outside these 42 has not been validated against
Validation month: 2026-03 only -- no other month has been checked
Precision@50 is the only k this repo defends; ranks below roughly 50-100 aren't individually validated against a held-out label.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any `refresh` row, a person checks:**
- `update_date_unknown` rows: is this page actually stale, or just missing an update-date
  record? The queue can't tell the difference — a plain data gap and a genuinely stale page
  produce the identical code.
- `recently_updated` rows still ranked high: per ML-07's own top-10 review, a CTR gap on a page
  updated a few days ago may just need more time to show up in search, not another rewrite.
- Any `deep`/`page_3_5` position-tier row: the expected-CTR baseline down there is already tiny
  (ML-06: 0.07% at `deep`), so a numerically real gap can still be practically small.

**No-go list — never automate:**
- Publishing, unpublishing, or deprioritizing a page directly from this queue's action label.
- Treating `refresh` as a ranked promise of recovery — it's a review priority, not a guarantee
  (ML-09 Section 4).
- Exporting anything beyond `client_hash_id`/`content_hash_id` — no titles, URLs, domains, or
  raw queries leave `work/outputs/` (`DATA_USE.md`).
- Re-ranking by `rf_probability` instead of `baseline_score` — that would silently swap in the
  weaker-tested ranker ML-08 already found underperforms.

In [11]:
# Human-review triggers, counted on the actual queue rather than described in the abstract
n_unknown = final_queue["reason_codes"].str.contains("update_date_unknown").sum()
n_recent = final_queue["reason_codes"].str.contains("recently_updated").sum()
n_shallow_tier = final_queue["position_bucket"].isin(["deep", "page_3_5"]).sum()

print(f"Rows flagged update_date_unknown (needs a manual staleness check): {n_unknown:,}")
print(f"Rows flagged recently_updated but still ranked (may just need time, not another refresh): {n_recent:,}")
print(f"Rows in the deep/page_3_5 tiers (small-absolute-gap risk): {n_shallow_tier:,}")

# No-go check: confirm the export column list carries only hashed IDs, never anything client-identifying
export_cols = ["rank", "client_hash_id", "content_hash_id", "action", "confidence", "reason_codes",
               "baseline_score", "rf_probability", "position_bucket", "ctr_prev30", "expected_ctr",
               "days_since_update", "imp_prev30", "is_declining"]
BANNED_IN_EXPORT = ("title", "url", "domain", "query", "name", "email")
hits = [c for c in export_cols if any(bad in c.lower() for bad in BANNED_IN_EXPORT)]
print(f"\nClient-identifying columns in the export list: {hits or 'none'}")

Rows flagged update_date_unknown (needs a manual staleness check): 41,840
Rows flagged recently_updated but still ranked (may just need time, not another refresh): 12,548
Rows in the deep/page_3_5 tiers (small-absolute-gap risk): 27,336

Client-identifying columns in the export list: none


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Base rate drift.** The held-out base rate is 0.320 (ML-05/07/08/09, consistent run to run).
  A fresh month landing meaningfully outside that would mean the underlying decline dynamics
  shifted and the rule needs re-checking before being trusted again.
- **Precision@50 falling out of its known range.** Every honest run in this repo has landed
  0.44-0.56 for the baseline; a fresh month scoring materially below that is the signal to stop
  trusting the queue until re-validated, not to keep shipping it.
- **The staleness data-quality issue resolving.** If `content_updated_date` in `dim_content`
  ever starts reflecting true point-in-time state instead of a current-state snapshot, the
  `365+` staleness bucket ML-07 found empty becomes usable — a real trigger to rebuild the
  rule's staleness term, not just a curiosity.
- **Client mix changing.** The 29/13 client-grouped split is fixed to today's warehouse. A large
  batch of new clients added to the warehouse means the held-out validation no longer covers
  the current population and should be re-run.
- **The baseline/model gap reversing.** If a future Random Forest run beats the baseline's
  Precision@50 by a real margin (not the library-version noise ML-09 documented), that's the
  trigger to reopen the "which ranker is primary" decision this section made.

In [12]:
# This run's numbers next to the known reference range -- what a future run should be compared against
this_run_base_rate_full = data["is_declining"].mean()
this_run_p50_full = final_queue.sort_values("baseline_score", ascending=False).head(50)["is_declining"].mean()

test_client_ids = set(groups.iloc[test_idx])
test_queue = final_queue[final_queue["client_hash_id"].isin(test_client_ids)]
this_run_base_rate_holdout = test_queue["is_declining"].mean()
this_run_p50_holdout = test_queue.sort_values("baseline_score", ascending=False).head(50)["is_declining"].mean()

print(f"This run -- full-slice base rate: {this_run_base_rate_full:.3f} (reference: 0.285)")
print(f"This run -- held-out base rate: {this_run_base_rate_holdout:.3f} (reference: 0.320)")
print(f"This run -- full-slice Precision@50: {this_run_p50_full:.3f} (reference range: 0.44-0.56)")
print(f"This run -- held-out Precision@50: {this_run_p50_holdout:.3f} (reference range: 0.44-0.54)")
print("A future run landing outside these ranges is the trigger to stop trusting the queue "
      "until re-validated -- not a reason to keep shipping it unchanged.")

This run -- full-slice base rate: 0.285 (reference: 0.285)
This run -- held-out base rate: 0.320 (reference: 0.320)
This run -- full-slice Precision@50: 0.560 (reference range: 0.44-0.56)
This run -- held-out Precision@50: 0.540 (reference range: 0.44-0.54)
A future run landing outside these ranges is the trigger to stop trusting the queue until re-validated -- not a reason to keep shipping it unchanged.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Writes the full ranked queue to `work/outputs/action_playbook_queue.csv` (hashed IDs only, no
client-identifying fields — `DATA_USE.md`) and two small charts to `work/figures/` for the
paper: the action mix and the confidence mix.

In [13]:
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

export_cols = ["rank", "client_hash_id", "content_hash_id", "action", "confidence", "reason_codes",
               "baseline_score", "rf_probability", "position_bucket", "ctr_prev30", "expected_ctr",
               "days_since_update", "imp_prev30", "is_declining"]
final_queue[export_cols].to_csv("work/outputs/action_playbook_queue.csv", index=False)
print(f"Wrote {len(final_queue):,} rows to work/outputs/action_playbook_queue.csv")

action_counts = final_queue["action"].value_counts()
plt.figure(figsize=(5, 3))
action_counts.plot(kind="bar", color="#426B69")
plt.title("Action mix")
plt.ylabel("content items")
plt.tight_layout()
plt.savefig("work/figures/action_mix.png", dpi=150)
plt.close()

confidence_counts = final_queue["confidence"].value_counts().reindex(
    ["high", "medium", "low", "not_scored"], fill_value=0
)
plt.figure(figsize=(5, 3))
confidence_counts.plot(kind="bar", color="#6F4E7C")
plt.title("Confidence mix")
plt.ylabel("content items")
plt.tight_layout()
plt.savefig("work/figures/confidence_mix.png", dpi=150)
plt.close()

print("Wrote work/figures/action_mix.png and work/figures/confidence_mix.png")

Wrote 146,253 rows to work/outputs/action_playbook_queue.csv
Wrote work/figures/action_mix.png and work/figures/confidence_mix.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.